In [88]:
import os
import torch
import torch.nn as nn
import torchvision
from torchinfo import summary
from torch.utils.data import DataLoader
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

In [89]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 32
transforms = torchvision.transforms.Compose([
    torchvision.transforms.ToTensor(),
])

print(f"Using device {device}")

train_data = torchvision.datasets.MNIST(root="mnist", train=True, download=True, transform=transforms)
test_data = torchvision.datasets.MNIST(root="mnist", train=False, download=True, transform=transforms)

train_dataloader = DataLoader(train_data, batch_size=batch_size, shuffle=True, pin_memory=True, num_workers=4)
test_dataloader = DataLoader(test_data, batch_size=batch_size, shuffle=False, pin_memory=True, num_workers=4)

Using device cuda


In [90]:
class Encoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1, stride=1),    # (1, 28, 28) -> (32, 28, 28)
            nn.MaxPool2d(kernel_size=2, stride=2),    # (32, 28, 28) -> (32, 14, 14)
            nn.ReLU(),

            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1, stride=1),    # (32, 14, 14) -> (64, 14, 14)
            nn.MaxPool2d(kernel_size=2, stride=2),    # (64, 14, 14) -> (64, 7, 7)
            nn.ReLU(),
        )

        self.mu_layer = nn.Linear(64*7*7, latent_dim)    # (1, 64*7*7) -> (1, latent_dim)
        self.logvar_layer = nn.Linear(64*7*7, latent_dim)    # (1, 64*7*7) -> (1, latent_dim)
    
    def forward(self, img):
        x = self.conv_layers(img)
        x = x.view(x.size(0), -1)    # (64, 7, 7) -> (1, 64*7*7)

        mu = self.mu_layer(x)
        logvar = self.logvar_layer(x)

        return mu, logvar

In [91]:
class Decoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()

        self.fcnn_layer = nn.Linear(latent_dim, 64*7*7)    # (1, latent_dim) -> (1, 64*7*7)

        self.conv_layers = nn.Sequential(
            nn.ConvTranspose2d(in_channels=64, out_channels=32, kernel_size=4, padding=1, stride=2),    # (64, 7, 7) -> (32, 14, 14)
            nn.ReLU(),

            nn.ConvTranspose2d(in_channels=32, out_channels=1, kernel_size=4, padding=1, stride=2),    # (32, 14, 14) -> (3, 28, 28)
            nn.Sigmoid(),
        )
    
    def forward(self, latent):
        x = self.fcnn_layer(latent)
        x = x.view(x.size(0), 64, 7, 7)    # (1, 64*7*7) -> (64, 7, 7)

        img = self.conv_layers(x)
        return img
    

In [92]:
class VAE(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.encoder = Encoder(latent_dim)
        self.decoder = Decoder(latent_dim)
    
    def forward(self, img):
        mu, logvar = self.encoder(img)

        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        z = mu + eps * std

        out = self.decoder(z)
        return out, mu, logvar

In [93]:
class VAELoss(nn.Module):
    def __init__(self, beta):
        super().__init__()
        self.beta = beta
        self.mse_loss = nn.MSELoss(reduction="sum")
    
    def forward(self, img, out, mu, logvar):
        reconstruction_loss = self.mse_loss(img, out)
        kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())

        loss = (reconstruction_loss + self.beta * kl_loss) / img.shape[0]
        reconstruction_loss /= img.shape[0]
        kl_loss /= img.shape[0]
        
        return loss, reconstruction_loss, kl_loss

In [94]:
def train_one_epoch(model, train_dataloader, test_dataloader, loss_func, optimizer, device):
    train_loss = 0
    train_reconstruction_loss = 0
    train_kl_loss = 0

    model.train()
    for img, label in tqdm(train_dataloader, desc="Training"):
        img = img.to(device)
        label = label.to(device)

        out, mu, logvar = model(img)

        optimizer.zero_grad()
        loss, reconstruction_loss, kl_loss = loss_func(img, out, mu, logvar)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        train_reconstruction_loss += reconstruction_loss.item()
        train_kl_loss += kl_loss.item()
    
    test_loss = 0
    test_reconstruction_loss = 0
    test_kl_loss = 0

    model.eval()
    with torch.no_grad():
        for img, label in tqdm(test_dataloader, desc="Validating"):
            img = img.to(device)
            label = label.to(device)

            out, mu, logvar = model(img)
            loss, reconstruction_loss, kl_loss = loss_func(img, out, mu, logvar)

            test_loss += loss.item()
            test_reconstruction_loss += reconstruction_loss.item()
            test_kl_loss += kl_loss.item()
    
    train_loss /= len(train_dataloader)  
    train_reconstruction_loss /= len(train_dataloader)  
    train_kl_loss /= len(train_dataloader)  
    test_loss /= len(test_dataloader)  
    test_reconstruction_loss /= len(test_dataloader)   
    test_kl_loss /= len(test_dataloader)

    return train_loss, train_reconstruction_loss, train_kl_loss, test_loss, test_reconstruction_loss, test_kl_loss


In [99]:
def visualize(model, test_dataloader, epoch, device):
    sample_in = []
    sample_out = []
    all_mu = []
    all_labels = []

    model.eval()
    with torch.no_grad():
        for img, label in tqdm(test_dataloader, desc="Visualizing"):
            img = img.to(device)
            label = label.to(device)

            out, mu, logvar = model(img)

            if (len(sample_in) == 0):    # The first batch will be visualized for reconstruction
                sample_in.append(img)
                sample_out.append(out)
            
            all_mu.append(mu)
            all_labels.append(label)
    
    sample_in = torch.cat(sample_in, dim=0)
    sample_out = torch.cat(sample_out, dim=0)
    all_mu = torch.cat(all_mu, dim=0)
    all_labels = torch.cat(all_labels, dim=0)

    fig, axes = plt.subplots(5, 2, figsize=(6, 12))
    for i in range(5):
        axes[i, 0].imshow(sample_in[i].squeeze(0).cpu())
        axes[i, 0].set_title("input")
        axes[i, 1].imshow(sample_out[i].squeeze(0).cpu())
        axes[i, 1].set_title("output")
    
    plt.tight_layout()
    plt.savefig(f"results/reconstruction_epoch_{epoch}.png")
    plt.close()

    pca = PCA(n_components=2)
    all_mu_2d = pca.fit_transform(all_mu.cpu().numpy())
    
    plt.figure(figsize=(8,6)) 
    scatter = plt.scatter(all_mu_2d[:,0], all_mu_2d[:,1], c=all_labels.cpu(), cmap="tab10",s=5)

    cbar = plt.colorbar(scatter, ticks=np.arange(10))
    cbar.ax.set_yticklabels([f"{i}" for i in range(10)])
    plt.savefig(f"results/latent_space_{epoch}.png")
    plt.close()
    


In [100]:
def train(num_epochs, model, train_dataloader, test_dataloader, loss_func, optimizer, device, patience):
    train_losses = []
    train_reconstruction_losses = []
    train_kl_losses = []
    test_losses = []
    test_reconstruction_losses = []
    test_kl_losses = []

    best_test_loss = float("inf")
    patience_count = 0
    for epoch in range(num_epochs):
        train_loss, train_reconstruction_loss, train_kl_loss, test_loss, test_reconstruction_loss, test_kl_loss = train_one_epoch(model, train_dataloader, test_dataloader, loss_func, optimizer, device)

        train_losses.append(train_loss)
        train_reconstruction_losses.append(train_reconstruction_loss)
        train_kl_losses.append(train_kl_loss)
        test_losses.append(test_loss)
        test_reconstruction_losses.append(test_reconstruction_loss)
        test_kl_losses.append(test_kl_loss)

        visualize(model, test_dataloader, epoch, device)

        print(f"Epoch {epoch} | train_loss: {train_loss:.2f} | train_recon: {train_reconstruction_loss:.2f} | train_kl: {train_kl_loss:.2f} | test_loss: {test_loss:.2f}")

        if (test_loss < best_test_loss):
            patience_count = 0
            best_test_loss = test_loss
            torch.save(model.state_dict(), "results/best_model.pt")
        else:
            patience_count += 1
            if (patience_count >= patience):
                print("Early stopping")
                break

    return train_losses, train_reconstruction_losses, train_kl_losses, test_losses, test_reconstruction_losses, test_kl_losses


In [101]:
latent_dim = 32
model = VAE(latent_dim)
model = model.to(device)

summary(model, (16, 1, 28, 28))

Layer (type:depth-idx)                   Output Shape              Param #
VAE                                      [16, 1, 28, 28]           --
├─Encoder: 1-1                           [16, 32]                  --
│    └─Sequential: 2-1                   [16, 64, 7, 7]            --
│    │    └─Conv2d: 3-1                  [16, 32, 28, 28]          320
│    │    └─MaxPool2d: 3-2               [16, 32, 14, 14]          --
│    │    └─ReLU: 3-3                    [16, 32, 14, 14]          --
│    │    └─Conv2d: 3-4                  [16, 64, 14, 14]          18,496
│    │    └─MaxPool2d: 3-5               [16, 64, 7, 7]            --
│    │    └─ReLU: 3-6                    [16, 64, 7, 7]            --
│    └─Linear: 2-2                       [16, 32]                  100,384
│    └─Linear: 2-3                       [16, 32]                  100,384
├─Decoder: 1-2                           [16, 1, 28, 28]           --
│    └─Linear: 2-4                       [16, 3136]                103

In [102]:
os.makedirs("results", exist_ok=True)
num_epochs = 1
beta = 1
lr = 1e-4
patience = 5
loss_func = VAELoss(beta)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

train_losses, train_reconstruction_losses, train_kl_losses, test_losses, test_reconstruction_losses, test_kl_losses = train(num_epochs, model, train_dataloader, test_dataloader, loss_func, optimizer, device, patience)

Visualizing: 100%|██████████| 313/313 [00:00<00:00, 376.53it/s]


Epoch 0 | train_loss: 56.19 | train_recon: 45.74 | train_kl: 10.45 | test_loss: 42.62
